# A3-M2-2 · FAISS 검색과 메타데이터
교안의 같은 번호 단원과 함께 봅니다. 노트북은 01 → 02 → 03 순서로 실행합니다.

## 02-1. 준비 — CSV와 저장한 벡터 불러오기

In [1]:
import json, os
import numpy as np
import pandas as pd
import faiss
from emb import embed, EMB_MODEL

faq = pd.read_csv("shop_faq.csv")
faq["doc"] = faq["question"] + " " + faq["answer"]
X = np.load("faq_vectors.npy")
print("FAQ", len(faq), "건 | 벡터", X.shape)

FAQ 20 건 | 벡터 (20, 384)


## 02-2. IndexFlatIP 만들고 검색하기

In [2]:
index = faiss.IndexFlatIP(X.shape[1])
index.add(X)
print("저장된 개수:", index.ntotal)

q = embed(["택배 언제 와요?"], kind="query")
D, I = index.search(q, 3)
print("D:", D.round(3))
print("I:", I)
print(faq.loc[I[0], ["faq_id", "category", "question"]].assign(score=D[0].round(3)).to_string(index=False))

저장된 개수: 20


c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 822.97it/s]


D: [[0.846 0.827 0.824]]
I: [[0 8 2]]
 faq_id category                question  score
    101       배송             배송은 언제 오나요?  0.846
    301     교환반품      교환이나 반품은 언제까지 되나요?  0.827
    103       배송 주문한 상품의 배송 조회는 어디서 하나요?  0.824


## 02-3. 질문 여러 개 · k가 문서 수보다 클 때

In [3]:
qs = ["환불 며칠 걸려요?", "포인트 언제 없어져요?"]
D, I = index.search(embed(qs, kind="query"), 2)
for q, d, i in zip(qs, D, I):
    print(q, "→", faq.loc[i, "question"].tolist(), d.round(3).tolist())

D, I = index.search(embed(["환불"], kind="query"), 25)      # FAQ는 20건
print("k=25 마지막 5칸 I:", I[0][-5:].tolist())

환불 며칠 걸려요? → ['환불은 언제 되나요?', '부분 환불도 되나요?'] [0.8849999904632568, 0.8709999918937683]
포인트 언제 없어져요? → ['회원 탈퇴는 어떻게 하나요?', '비밀번호를 잊어버렸어요.'] [0.8199999928474426, 0.8059999942779541]
k=25 마지막 5칸 I: [-1, -1, -1, -1, -1]


## 03-1. 메타데이터 구조 — faq_id를 FAISS id로

In [4]:
ids = faq["faq_id"].to_numpy(dtype="int64")
index = faiss.IndexIDMap2(faiss.IndexFlatIP(X.shape[1]))
index.add_with_ids(X, ids)

meta = faq.set_index("faq_id")                    # id → category, question, answer
D, I = index.search(embed(["택배 언제 와요?"], kind="query"), 3)
print("I:", I)                                    # 이제 FAQ 번호가 나온다
print(meta.loc[I[0], ["category", "question"]].assign(score=D[0].round(3)).to_string())
print("401번 벡터 앞 5칸:", index.reconstruct(401)[:5].round(3))

I: [[101 301 103]]
       category                 question  score
faq_id                                         
101          배송              배송은 언제 오나요?  0.846
301        교환반품       교환이나 반품은 언제까지 되나요?  0.827
103          배송  주문한 상품의 배송 조회는 어디서 하나요?  0.824
401번 벡터 앞 5칸: [ 0.04  -0.01  -0.03  -0.068  0.057]


## 04-1. 카테고리 필터 — 검색 안에서 거르기

In [5]:
def ids_of(categories):
    return meta.index[meta["category"].isin(categories)].to_numpy(dtype="int64")

qv = embed(["돈 언제 돌려줘요?"], kind="query")
for cats in [None, ["환불"], ["환불", "결제"], ["배송"]]:
    params = None if cats is None else faiss.SearchParameters(sel=faiss.IDSelectorBatch(ids_of(cats)))
    D, I = index.search(qv, 3, params=params)
    got = [(int(i), meta.loc[i, "category"], round(float(d), 3)) for d, i in zip(D[0], I[0]) if i != -1]
    print("전체" if cats is None else cats, got)

전체 [(301, '교환반품', 0.847), (202, '결제', 0.838), (701, '적립금', 0.837)]
['환불'] [(401, '환불', 0.836), (402, '환불', 0.823)]
['환불', '결제'] [(202, '결제', 0.838), (401, '환불', 0.836), (201, '결제', 0.828)]
['배송'] [(101, '배송', 0.835), (105, '배송', 0.829), (102, '배송', 0.825)]


## 04-2. 검색 뒤에 거르면 왜 안 되나 · 빼고 찾기

In [6]:
D, I = index.search(qv, 3)
after = [int(i) for i in I[0] if meta.loc[i, "category"] == "결제"]
print("검색 후 거르기(결제):", after, "→", len(after), "개")

params = faiss.SearchParameters(sel=faiss.IDSelectorBatch(ids_of(["결제"])))
D, I = index.search(qv, 3, params=params)
print("검색 안에서 거르기(결제):", I[0].tolist())

keep = faiss.IDSelectorBatch(ids_of(["환불"]))
params = faiss.SearchParameters(sel=faiss.IDSelectorNot(keep))      # 환불만 빼고
D, I = index.search(qv, 3, params=params)
print("환불 빼고:", [(int(i), meta.loc[i, "category"]) for i in I[0]])

검색 후 거르기(결제): [202] → 1 개
검색 안에서 거르기(결제): [202, 201, 203]
환불 빼고: [(301, '교환반품'), (202, '결제'), (701, '적립금')]


## 05-1. 검색 함수 하나로 묶기

In [7]:
def search(question, k=3, categories=None):
    params = None
    if categories:
        params = faiss.SearchParameters(sel=faiss.IDSelectorBatch(ids_of(categories)))
    D, I = index.search(embed([question], kind="query"), k, params=params)
    hits = [(int(i), float(d)) for d, i in zip(D[0], I[0]) if i != -1]
    res = meta.loc[[i for i, _ in hits], ["category", "question", "answer"]].copy()
    res.insert(0, "score", [round(s, 3) for _, s in hits])
    return res

print(search("카드 할부 돼요?")[["score", "category", "question"]].to_string())
print()
print(search("카드 할부 돼요?", categories=["결제"])[["score", "category", "question"]].to_string())

        score category         question
faq_id                                 
202     0.876       결제     무이자 할부가 되나요?
602     0.827       회원  회원 탈퇴는 어떻게 하나요?
401     0.826       환불      환불은 언제 되나요?

        score category            question
faq_id                                    
202     0.876       결제        무이자 할부가 되나요?
201     0.826       결제  어떤 결제 수단을 쓸 수 있나요?
203     0.802       결제   현금영수증은 어떻게 발급하나요?


## 05-2. 저장하고 다시 열기 — 인덱스 + 메타데이터 + 설정

In [8]:
os.makedirs("faq_store", exist_ok=True)
faiss.write_index(index, "faq_store/index.faiss")
meta.drop(columns="doc").to_csv("faq_store/meta.csv", encoding="utf-8-sig")
manifest = {"model": EMB_MODEL, "dim": int(X.shape[1]), "count": int(index.ntotal), "metric": "cosine(IP)"}
json.dump(manifest, open("faq_store/manifest.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
for f in sorted(os.listdir("faq_store")):
    print(f"{f:<14} {os.path.getsize('faq_store/' + f):>7,} bytes")

ix2 = faiss.read_index("faq_store/index.faiss")
meta2 = pd.read_csv("faq_store/meta.csv").set_index("faq_id")
man = json.load(open("faq_store/manifest.json", encoding="utf-8"))
assert man["model"] == EMB_MODEL and ix2.ntotal == len(meta2)
D, I = ix2.search(embed(["반품 기간"], kind="query"), 2)
print("다시 연 인덱스:", I[0].tolist(), meta2.loc[I[0], "question"].tolist())

index.faiss     30,970 bytes
manifest.json      103 bytes
meta.csv         3,171 bytes
다시 연 인덱스: [301, 401] ['교환이나 반품은 언제까지 되나요?', '환불은 언제 되나요?']
